In [5]:
import requests
import pandas as pd
import time
from datetime import date
from pathlib import Path

In [4]:
def fetch_all(base="https://amahaho.com"):
    rows = []
    page = 1

    while True:
        r = requests.get(f"{base}/products.json",
                         params={"limit": 250, "page": page})
        products = r.json()["products"]

        if not products:          # empty page = we've reached the end
            break

        for p in products:
            v = p["variants"][0]   # first variant = the default price
            rows.append({
                "title":     p["title"],
                "category":  p.get("product_type") or "uncategorised",
                "price_rwf": float(v["price"]),
                "in_stock":  v["available"],
            })

        print(f"page {page}: {len(products)} products")
        page += 1
        time.sleep(1)              # be polite - one request per second

    return pd.DataFrame(rows)


df = fetch_all()
print(f"\ntotal: {len(df)} products")
df.head()

page 1: 250 products
page 2: 234 products

total: 484 products


,title,category,price_rwf,in_stock
0,Pork bones meat/kg,uncategorised,8500.0,False
1,"Samsung A17 4G 128GB,4GB RAM,2years warranty",uncategorised,272200.0,True
2,"Samsung A16 4G 128GB, 4GB RAM, 2yr warranty",Smartphone,250000.0,True
3,"Samsung A07 4G 128GB, 4GB RAM 2yr warranty",Smartphone,200000.0,True
4,"Samsung A07 4G 64GB, 4GB RAM, 2yr warranty",Smartphone,172000.0,True


In [6]:
Path("data").mkdir(exist_ok=True)
HISTORY = Path("data/price_history.csv")

df["date"] = date.today().isoformat()
df["shop"] = "amahaho"

# add today's scrape to whatever we've collected before
if HISTORY.exists():
    combined = pd.concat([pd.read_csv(HISTORY), df])
else:
    combined = df

# one row per product per shop per day - re-running today won't duplicate
combined = combined.drop_duplicates(subset=["date", "shop", "title"], keep="last")
combined.to_csv(HISTORY, index=False)

print(f"{len(combined)} rows covering {combined.date.nunique()} day(s)")

484 rows covering 1 day(s)
